# 09 — Histology classical ML (SVM / RF / KNN)

Same `train_classical_cv` pipeline as notebook 03 but on LC25000 features. Per-fold JSON files saved under `results/`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import yaml

from utils.seed import set_seed
from utils.models import CLASSICAL_REGISTRY
from utils.training import train_classical_cv, save_fold_results
from utils.metrics import aggregate_folds
from utils.stats import format_results_table

with open('../configs/histology.yaml') as f:
    cfg = yaml.safe_load(f)
set_seed(cfg['seed'])

cache_dir = Path('..') / cfg['data']['cache_dir']
data = np.load(cache_dir / 'features.npz', allow_pickle=True)
X, y = data['X'], data['y']
print('X', X.shape, 'pos rate:', y.mean())

X (15000, 208) pos rate: 0.6666666666666666


In [2]:
results_dir = Path('..') / cfg['paths']['results']
results_dir.mkdir(parents=True, exist_ok=True)

aggregated = {}
for name, (builder, grid) in CLASSICAL_REGISTRY.items():
    print(f'\n=== {name} ===')
    folds = train_classical_cv(
        X, y, builder, grid,
        n_splits=cfg['cv']['n_splits'],
        inner_splits=cfg['cv']['inner_splits'],
        scoring=cfg['cv']['scoring'],
        seed=cfg['seed'],
        verbose=True,
    )
    save_fold_results(folds, results_dir / f'histology_{name}.json')
    aggregated[name] = aggregate_folds([f.metrics_calibrated for f in folds])

format_results_table(aggregated)


=== SVM ===
[fold 0] train=12000 test=3000
[fold 0] best_params={'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'} thr=0.958 AUC=1.000 BalAcc(cal)=0.998
[fold 1] train=12000 test=3000
[fold 1] best_params={'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'} thr=0.948 AUC=1.000 BalAcc(cal)=1.000
[fold 2] train=12000 test=3000
[fold 2] best_params={'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'} thr=0.947 AUC=1.000 BalAcc(cal)=1.000
[fold 3] train=12000 test=3000
[fold 3] best_params={'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'} thr=0.938 AUC=1.000 BalAcc(cal)=1.000
[fold 4] train=12000 test=3000
[fold 4] best_params={'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'} thr=0.757 AUC=1.000 BalAcc(cal)=1.000

=== RF ===
[fold 0] train=12000 test=3000
[fold 0] best_params={'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100} thr=0.810 AUC=1.000 BalAcc(cal)=0.999
[fold 1] train=12000 test=3000
[fold 1] best_params={'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100} thr=0.83

,accuracy,sensitivity,specificity,precision,f1,balanced_accuracy,auc_roc,pr_auc
model,,,,,,,,
SVM,0.999 ± 0.001,0.999 ± 0.002,1.000 ± 0.000,1.000 ± 0.000,0.999 ± 0.001,0.999 ± 0.001,1.000 ± 0.000,1.000 ± 0.000
RF,1.000 ± 0.001,0.999 ± 0.001,1.000 ± 0.000,1.000 ± 0.000,1.000 ± 0.001,1.000 ± 0.001,1.000 ± 0.000,1.000 ± 0.000
KNN,1.000 ± 0.000,1.000 ± 0.000,1.000 ± 0.000,1.000 ± 0.000,1.000 ± 0.000,1.000 ± 0.000,1.000 ± 0.000,1.000 ± 0.000
